# Figure 3.6 with H²: bandwidth distributions for increasing sample size

This standalone notebook adapts the increasing-sample-size experiment from Figure 3.6 to **squared Hellinger divergence**. It produces one figure with ten separate panels for $n=100,200,\ldots,1000$, with the same fixed geometric bandwidth grid at every sample size.

For Gaussian KDEs $p=\widehat f_{h_i}$ and $q=\widehat f_{h_j}$, use

$$H^2(p,q)=\frac12\int_{\mathbb R}(\sqrt p-\sqrt q)^2\,dx,$$

$$S_{ij}=\exp(-\lambda H^2(p,q)),\qquad \lambda=1.$$

The selected grid bandwidth maximizes the Perron vector of $S$. H² is integrated with the same composite trapezoidal rule and finite interval as in the earlier divergence notebook. No other divergence is computed.

**Run the cells in order.** The default `"thesis"` profile uses $M=1000$ repetitions per sample size and $N=201$ bandwidth candidates. Checkpoints allow interruption and resume. Matching H² choices from the earlier notebook can be imported automatically; the source files are read without modification. The `"preview"` profile uses $M=20$, $N=21$ for exploration.

Sources: [Figure 3.6, `fig:distribution_of_chosen_bandwidths_for_increasing_sample_size`, and Hellinger definition in the alpha-divergence subsection](../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [earlier Gaussian divergence experiment](gaussian_symmetric_divergence_bandwidth_distributions.ipynb); [shared quadrature](kde_tools/quadrature.py); [shared Perron-vector selector](kde_tools/similarities.py).

All spectral choices use the centralized stabilized Lanczos solver in `kde_tools.perron`; checkpoint settings record its version.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import sys
import time
from pathlib import Path

working_directory = Path.cwd().resolve()
code_directory = next(
    (candidate.resolve()
     for root in (working_directory, *working_directory.parents)
     for candidate in (root, root / "Code" / "Code_kde")
     if (candidate / "kde_tools" / "pdf_export.py").is_file()),
    None,
)
if code_directory is None:
    raise FileNotFoundError("Run this notebook from the project or Code/Code_kde folder.")
if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))
os.environ.setdefault("MPLCONFIGDIR", str(code_directory / ".matplotlib-cache"))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

from kde_tools.bandwidths import bandwidth_grid_linear_in_psi_beta, gaussian_mise_oracle_bandwidth
from kde_tools.density import gaussian_kde_values
from kde_tools.perron import PERRON_SOLVER_VERSION
from kde_tools.pdf_export import enable_automatic_pdf_saving
from kde_tools.quadrature import quadrature_grid_for_kdes, trapezoidal_integral
from kde_tools.similarities import spectral_choice_from_divergence

notebook_name = "gaussian_hellinger_bandwidth_distributions_vs_n"
pdf_output_directory = enable_automatic_pdf_saving(
    notebook_name, output_root=code_directory / "outputs" / "pdfs"
)
data_output_directory = code_directory / "outputs" / "data" / notebook_name
image_output_directory = code_directory / "outputs" / "images" / notebook_name
existing_data_directory = (
    code_directory / "outputs" / "data" / "gaussian_symmetric_divergence_bandwidth_distributions"
)
print(f"Automatic PDF exports: {pdf_output_directory}")



## Settings

Sample from $N(0,1)$ for each $n=100,200,\ldots,1000$. Keep $h_{\min}=0.01$, $h_{\max}=2$, $\lambda=1$, 6,001 spatial nodes, and padding equal to six times the largest bandwidth on each side of the sample range. The fixed bandwidth grid is geometric.

Use the seed `base_seed + repetition` at every sample size, so each repetition uses nested prefixes of one Gaussian sample stream. Only $n$ changes within a profile. The optional dashed Gaussian MISE line is a reference; it is not an H²-optimal bandwidth.

Sources: [earlier experiment settings and sample-size sweep](gaussian_symmetric_divergence_bandwidth_distributions.ipynb); [geometric grid and optional Gaussian MISE reference](kde_tools/bandwidths.py); [quadrature interval](kde_tools/quadrature.py).


In [ ]:
RUN_PROFILE = "thesis"  # "thesis" or "preview"
PROFILES = {
    "thesis": {"repetitions": 1000, "grid_size": 201},
    "preview": {"repetitions": 20, "grid_size": 21},
}
SAMPLE_SIZES = list(range(100, 1001, 100))
REUSE_CHECKPOINTS = True
IMPORT_MATCHING_EXISTING_H2 = True
SHOW_MISE_REFERENCE = True

settings = {
    "perron_solver": PERRON_SOLVER_VERSION,
    **PROFILES[RUN_PROFILE],
    "sigma2": 1.0,
    "lambda_value": 1.0,
    "h_min": 0.01,
    "h_max": 2.0,
    "m_x": 6001,
    "padding": 6.0,
    "base_seed": 910_000,
}
if settings["repetitions"] < 1 or settings["grid_size"] < 2 or settings["m_x"] < 2:
    raise ValueError("Repetitions must be positive and grids need at least two points.")
if settings["sigma2"] <= 0 or settings["lambda_value"] <= 0 or settings["padding"] <= 0:
    raise ValueError("Variance, lambda, and padding must be positive.")
h_grid = bandwidth_grid_linear_in_psi_beta(
    h_min=settings["h_min"], h_max=settings["h_max"], beta=2.0,
    grid_size=settings["grid_size"],
)
configuration = {
    "settings": settings, "sample_sizes": SAMPLE_SIZES, "method": "H2",
    "h2_implementation_version": 1,
    "sampling": "normal_prefixes_from_base_seed_plus_repetition",
}
configuration_id = hashlib.sha256(json.dumps(configuration, sort_keys=True).encode()).hexdigest()[:12]
run_directory = data_output_directory / RUN_PROFILE / configuration_id
image_directory = image_output_directory / RUN_PROFILE / configuration_id
for directory in (run_directory, image_directory):
    directory.mkdir(parents=True, exist_ok=True)
(run_directory / "settings.json").write_text(
    json.dumps({"profile": RUN_PROFILE, **configuration}, indent=2) + "\n"
)
print(f"H² only; sample sizes: {SAMPLE_SIZES}")
print(f"Profile: {RUN_PROFILE}; M={settings['repetitions']} per n; "
      f"N={len(h_grid)}; lambda={settings['lambda_value']:g}")
print(f"Checkpoints and data: {run_directory}")


## H² calculation and checkpoint reuse

For the trapezoidal weights $w_k$ and KDE values $p_i(x_k)$, compute

$$m_i=\sum_k w_k p_i(x_k),\qquad A_{ij}=\sum_k w_k\sqrt{p_i(x_k)p_j(x_k)},$$

$$H^2_{ij}\approx\tfrac12(m_i+m_j)-A_{ij}.$$

This is the algebraic expansion of the H² definition above on the finite quadrature interval. It follows the earlier notebook: densities are not renormalized on that interval, small negative roundoff is clipped, and the diagonal is set to zero. The maximum KDE mass error is stored for each repetition as a diagnostic.

The checkpoint importer checks the sample size, repetition count, seed, variance, bandwidth grid, lambda, spatial nodes, padding, and known source implementation version. It can reuse the H² column from either the earlier single-size run or its three-divergence sample-size sweep. Imported choices are copied into this notebook's own checkpoint.

Sources: [Hellinger definition and composite trapezoidal rule](../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [the earlier notebook's `compute_divergences` and `compute_n_sweep_divergences`](gaussian_symmetric_divergence_bandwidth_distributions.ipynb).


In [ ]:
def compute_h2_divergence(samples, h_grid, *, m_x, padding):
    """Raw squared Hellinger divergence, matching the earlier notebook."""
    x_grid = quadrature_grid_for_kdes(samples, h_grid, m_x=m_x, padding=padding)
    densities = gaussian_kde_values(samples, h_grid, x_grid)
    masses = trapezoidal_integral(densities, x_grid, axis=1)
    weights = np.full(len(x_grid), (x_grid[-1] - x_grid[0]) / (len(x_grid) - 1))
    weights[[0, -1]] *= 0.5
    square_roots = np.sqrt(densities)
    affinity = (square_roots * weights[None, :]) @ square_roots.T
    divergence = np.maximum(0.5 * (masses[:, None] + masses[None, :]) - affinity, 0.0)
    divergence[:] = 0.5 * (divergence + divergence.T)
    np.fill_diagonal(divergence, 0.0)
    if not np.all(np.isfinite(divergence)) or np.any(divergence < 0):
        raise ArithmeticError("Invalid H² divergence matrix.")
    return divergence, float(np.max(np.abs(masses - 1.0)))


def validate_h2_checkpoint(indices, mass_errors, completed, repetitions, grid_size):
    if indices.shape != (repetitions,) or mass_errors.shape != (repetitions,):
        raise ValueError("Checkpoint array shapes differ from the current experiment.")
    if not np.issubdtype(indices.dtype, np.integer):
        raise ValueError("Checkpoint bandwidth indices must be integers.")
    if not 0 <= completed <= repetitions:
        raise ValueError("Invalid checkpoint completion count.")
    if np.any(indices[:completed] < 0) or np.any(indices[:completed] >= grid_size):
        raise ValueError("Checkpoint contains invalid bandwidth indices.")
    if not np.all(np.isfinite(mass_errors[:completed])):
        raise ValueError("Checkpoint contains invalid quadrature diagnostics.")


def find_existing_h2_checkpoint(point_settings, h_grid, source_directory, profile):
    """Read the most complete compatible H² column from the earlier experiment."""
    if source_directory is None:
        return None
    source_directory = Path(source_directory)
    n = point_settings["sample_size"]
    candidates = list(source_directory.glob(f"{profile}/*/checkpoint.npz"))
    candidates += list(source_directory.glob(
        f"increasing_sample_size/{profile}/*/n{n}/checkpoint.npz"
    ))
    best = None
    for path in candidates:
        with np.load(path, allow_pickle=False) as checkpoint:
            source_configuration = json.loads(str(checkpoint["configuration_json"].item()))
            source_settings = source_configuration.get("settings", source_configuration)
            if not all(source_settings.get(key) == value for key, value in point_settings.items()):
                continue
            # These are the known H² formulas in the original notebook.
            if source_settings.get("implementation_version") != 2:
                continue
            source_methods = source_configuration.get("methods")
            if source_methods is not None:
                if source_configuration.get("sweep_implementation_version") != 1:
                    continue
            else:
                manifest_path = path.parent / "settings.json"
                if not manifest_path.exists():
                    continue
                manifest = json.loads(manifest_path.read_text())
                if manifest.get("settings") != source_settings:
                    raise ValueError(f"Source manifest and checkpoint settings disagree: {path}")
                source_methods = manifest.get("methods", [])
            if "H2" not in source_methods:
                continue
            if "h_grid" in checkpoint and not np.array_equal(checkpoint["h_grid"], h_grid):
                continue
            source_indices = checkpoint["selected_indices"]
            if source_indices.shape != (point_settings["repetitions"], len(source_methods)):
                raise ValueError(f"Invalid source checkpoint array shape: {path}")
            indices = source_indices[:, source_methods.index("H2")].copy()
            mass_errors = checkpoint["grid_mass_errors"].copy()
            completed = int(checkpoint["completed"].item())
        validate_h2_checkpoint(indices, mass_errors, completed, point_settings["repetitions"], len(h_grid))
        if best is None or completed > best["completed"]:
            best = {"indices": indices, "mass_errors": mass_errors,
                    "completed": completed, "path": str(path)}
    return best


def run_h2_point(sample_size, settings, h_grid, output_directory, *,
                 reuse=True, source_directory=None, profile="thesis"):
    """Compute or resume H² bandwidth selections for one sample size."""
    point_settings = {**settings, "sample_size": int(sample_size)}
    repetitions = int(point_settings["repetitions"])
    if repetitions < 1 or sample_size < 1:
        raise ValueError("Repetitions and sample_size must be positive.")
    point_json = json.dumps({
        "settings": point_settings, "method": "H2", "h2_implementation_version": 1,
    }, sort_keys=True)
    point_directory = Path(output_directory) / f"n{int(sample_size)}"
    point_directory.mkdir(parents=True, exist_ok=True)
    checkpoint_path = point_directory / "checkpoint.npz"
    indices = np.full(repetitions, -1, dtype=int)
    mass_errors = np.full(repetitions, np.nan)
    completed = 0
    imported_from = ""
    imported_completed = 0

    def save_checkpoint():
        temporary = checkpoint_path.with_suffix(".npz.tmp")
        with temporary.open("wb") as stream:
            np.savez_compressed(
                stream, configuration_json=point_json, h_grid=h_grid,
                selected_indices=indices, grid_mass_errors=mass_errors, completed=completed,
                imported_from=imported_from, imported_completed=imported_completed,
            )
        temporary.replace(checkpoint_path)

    if reuse and checkpoint_path.exists():
        with np.load(checkpoint_path, allow_pickle=False) as checkpoint:
            if str(checkpoint["configuration_json"].item()) != point_json:
                raise ValueError("H² checkpoint settings differ from the current configuration.")
            if not np.array_equal(checkpoint["h_grid"], h_grid):
                raise ValueError("H² checkpoint bandwidth grid differs from the current grid.")
            indices = checkpoint["selected_indices"].copy()
            mass_errors = checkpoint["grid_mass_errors"].copy()
            completed = int(checkpoint["completed"].item())
            imported_from = str(checkpoint["imported_from"].item())
            imported_completed = int(checkpoint["imported_completed"].item())
    elif reuse and source_directory is not None:
        previous = find_existing_h2_checkpoint(point_settings, h_grid, source_directory, profile)
        if previous is not None:
            indices = previous["indices"]
            mass_errors = previous["mass_errors"]
            completed = previous["completed"]
            imported_from = previous["path"]
            imported_completed = completed
            save_checkpoint()
            print(f"n={sample_size}: imported {completed}/{repetitions} H² choices from {imported_from}", flush=True)
    validate_h2_checkpoint(indices, mass_errors, completed, repetitions, len(h_grid))
    print(f"n={sample_size}: H² starting at {completed}/{repetitions}", flush=True)
    started = time.perf_counter()
    for repetition in range(completed, repetitions):
        rng = np.random.default_rng(point_settings["base_seed"] + repetition)
        samples = rng.normal(0.0, np.sqrt(point_settings["sigma2"]), size=int(sample_size))
        divergence, mass_errors[repetition] = compute_h2_divergence(
            samples, h_grid, m_x=point_settings["m_x"], padding=point_settings["padding"],
        )
        indices[repetition] = spectral_choice_from_divergence(
            divergence, point_settings["lambda_value"]
        )["selected_index"]
        completed = repetition + 1
        save_checkpoint()
        if completed == 1 or completed % 25 == 0 or completed == repetitions:
            print(f"n={sample_size}: H² {completed}/{repetitions}; elapsed this run: "
                  f"{time.perf_counter() - started:.1f}s", flush=True)
    validate_h2_checkpoint(indices, mass_errors, completed, repetitions, len(h_grid))
    counts = np.bincount(indices, minlength=len(h_grid))
    frequencies = counts / repetitions
    assert counts.sum() == repetitions and np.isclose(frequencies.sum(), 1.0)
    return {
        "sample_size": int(sample_size), "selected_indices": indices,
        "counts": counts, "frequencies": frequencies,
        "maximum_grid_mass_error": float(np.max(mass_errors)),
        "imported_from": imported_from, "imported_completed": imported_completed,
    }


## Repeated sampling

The next cell runs all ten sample sizes, resuming this notebook's checkpoints and importing matching earlier H² choices when enabled. The full thesis profile uses 1,000 repetitions per sample size. Run the final plotting cell after all ten sample sizes finish.


In [ ]:
results_by_n = {}
for current_n in SAMPLE_SIZES:
    results_by_n[current_n] = run_h2_point(
        current_n, settings, h_grid, run_directory,
        reuse=REUSE_CHECKPOINTS,
        source_directory=existing_data_directory if IMPORT_MATCHING_EXISTING_H2 else None,
        profile=RUN_PROFILE,
    )
print(f"Completed all {len(results_by_n)} H² distributions.")


## Saved data

Save the individual selections, frequencies on the fixed bandwidth grid, and one summary row per sample size. The summary also records the number of choices imported from an earlier run.


In [ ]:
selection_rows = []
frequency_rows = []
summary_rows = []
for current_n in SAMPLE_SIZES:
    result = results_by_n[current_n]
    chosen_h = h_grid[result["selected_indices"]]
    selection_rows.extend(
        {"n": current_n, "repetition": repetition,
         "seed": settings["base_seed"] + repetition, "method": "H2",
         "grid_index": int(index), "h": float(h_grid[index])}
        for repetition, index in enumerate(result["selected_indices"])
    )
    frequency_rows.extend(
        {"n": current_n, "method": "H2", "grid_index": index, "h": float(h),
         "count": int(result["counts"][index]),
         "relative_frequency": float(result["frequencies"][index])}
        for index, h in enumerate(h_grid)
    )
    modal_index = int(np.argmax(result["counts"]))
    summary_rows.append({
        "n": current_n, "method": "H2", "repetitions": settings["repetitions"],
        "modal_h": float(h_grid[modal_index]),
        "mean_h": float(np.mean(chosen_h)), "median_h": float(np.median(chosen_h)),
        "modal_frequency": float(result["frequencies"][modal_index]),
        "maximum_grid_mass_error": result["maximum_grid_mass_error"],
        "imported_completed": result["imported_completed"], "imported_from": result["imported_from"],
    })
for filename, rows in (
    ("selections.csv", selection_rows), ("frequencies.csv", frequency_rows),
    ("summary.csv", summary_rows),
):
    with (run_directory / filename).open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
print(f"CSV data: {run_directory}")
print(f"{'n':>5s} {'modal h':>10s} {'mean h':>10s} {'median h':>10s}")
for row in summary_rows:
    print(f"{row['n']:5d} {row['modal_h']:10.5f} {row['mean_h']:10.5f} {row['median_h']:10.5f}")


## Figure 3.6 with H²: ten separate distributions

Each panel shows one H² bandwidth-choice distribution. Sample sizes increase from left to right and then down. All panels use the same bandwidth positions and vertical limits. Geometric-grid positions are equally spaced and labeled with their bandwidth values, as in the earlier notebook.

The figure is displayed and saved automatically as a PDF under `outputs/pdfs/gaussian_hellinger_bandwidth_distributions_vs_n/`. A PNG copy is also saved. When enabled, dashed lines show the Gaussian MISE reference at each sample size, not an H²-optimal bandwidth.

Sources: [Figure 3.6 / increasing-sample-size experiment](../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [optional Gaussian MISE helper](kde_tools/bandwidths.py).


In [ ]:
if set(results_by_n) != set(SAMPLE_SIZES):
    raise RuntimeError("Finish all ten sample sizes before plotting.")
grid_positions = np.arange(len(h_grid))
ticks = np.unique(np.linspace(0, len(h_grid) - 1, 7, dtype=int))
y_max = 1.08 * max(np.max(result["frequencies"]) for result in results_by_n.values())
fig, axes = plt.subplots(5, 2, figsize=(14, 18), sharex=True, sharey=True)
for ax, current_n in zip(axes.flat, SAMPLE_SIZES):
    ax.bar(grid_positions, results_by_n[current_n]["frequencies"], width=0.86,
           color="#49A994", alpha=0.85, edgecolor="white", linewidth=0.25)
    if SHOW_MISE_REFERENCE:
        reference_h = np.sqrt(settings["sigma2"]) * gaussian_mise_oracle_bandwidth(current_n)
        if h_grid[0] <= reference_h <= h_grid[-1]:
            reference_position = np.interp(np.log(reference_h), np.log(h_grid), grid_positions)
            ax.axvline(reference_position, color="#1f77b4", linestyle="--", linewidth=1.4)
    ax.set_title(rf"$n={current_n}$", fontsize=13, pad=8)
    ax.set_xticks(ticks, [f"{h_grid[index]:.3g}" for index in ticks])
    ax.set_xlim(-0.7, len(h_grid) - 0.3)
    ax.set_ylim(0, y_max)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    ax.set_xlabel("selected bandwidth h", fontsize=10)
    ax.set_ylabel("relative frequency", fontsize=10)
    ax.tick_params(labelsize=9, colors="#304967", labelbottom=True, labelleft=True)
    ax.grid(axis="y", color="#DDE5EC", linewidth=0.7)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#B8C5D1")
fig.suptitle(
    r"$H^2$ spectral bandwidth distributions for increasing sample size" + "\n"
    + rf"Gaussian data; $M={settings['repetitions']}$ per $n$, $N={len(h_grid)}$, "
    + rf"$S=\exp(-\lambda H^2)$, $\lambda={settings['lambda_value']:g}$ ({RUN_PROFILE})",
    fontsize=16, y=0.995,
)
if SHOW_MISE_REFERENCE:
    fig.legend(
        handles=[Line2D([0], [0], color="#1f77b4", linestyle="--", linewidth=1.4)],
        labels=["Gaussian MISE reference for each sample size"],
        loc="lower center", frameon=False, fontsize=11, bbox_to_anchor=(0.5, 0.005),
    )
fig.tight_layout(rect=(0, 0.035, 1, 0.965), h_pad=2.0, w_pad=2.0)
png_path = image_directory / f"h2_increasing_n_M{settings['repetitions']}_N{len(h_grid)}.png"
fig.savefig(png_path, dpi=160, bbox_inches="tight")
plt.show()  # The shared automatic exporter saves this figure as a PDF before display.
plt.close(fig)
print(f"PNG: {png_path}")
print(f"PDF directory: {pdf_output_directory}")
